# 1. Research and Write an Article with a Sequential Crew

**Learning objectives**
- Define agents using `role`, `goal` and `backstory`.
- Define tasks using `description`, `expected_output` and `agent`.
- Understand sequential task context: Planner → Writer → Editor.
- Add a programmatic guardrail to validate an output.

**Workflow:** the Planner creates a content plan; the Writer uses that plan; the Editor reviews the draft.

# Setup: CrewAI + Google Gemini (Colab)

**Before running:** add a Gemini API key to Google Colab → Secrets with the name `GOOGLE_API_KEY` and enable notebook access to it. A Serper key is optional and is only needed for live web search in the relevant exercises.

These notebooks use the current CrewAI Python API style. The exact package versions are intentionally not pinned to an old 0.x release; after a successful lab setup, freeze the working environment for repeatability.

In [ ]:
%pip -q install -U "crewai[google-genai]" crewai-tools

In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Add GOOGLE_API_KEY in Colab Secrets and enable notebook access.")

llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.2
)
print("LLM configured.")

## Step 1 — Create the agents

Each agent has a defined responsibility. Keep the roles distinct so that collaboration adds value rather than duplicating work.

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Create a useful, balanced and evidence-aware plan about {topic}",
    backstory=(
        "You are an experienced content strategist. "
        "You identify the audience, key questions, themes and evidence "
        "needed for a clear educational article. Do not invent sources."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

writer = Agent(
    role="Content Writer",
    goal="Write a clear, engaging article using the approved content plan",
    backstory=(
        "You are a technical writer. You distinguish established facts "
        "from interpretation and do not introduce unsupported claims."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

editor = Agent(
    role="Editorial Reviewer",
    goal="Improve clarity, structure, consistency and factual caution",
    backstory=(
        "You are a careful editor. You remove repetition, flag unsupported "
        "claims and preserve the intended meaning."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

## Step 2 — Define the tasks

The tasks are listed in the order in which they should run. In a sequential crew, later tasks can use the outputs of earlier tasks as context.

In [ ]:
plan_task = Task(
    description=(
        "Prepare a content plan for an educational article on {topic}. "
        "Identify the intended audience, 4–6 key sections, important terms, "
        "questions to answer and claims that require verification. "
        "Do not claim to have searched the web unless a search tool was used."
    ),
    expected_output=(
        "A structured content plan with audience, article outline, key points, "
        "important terms and a list of claims requiring verification."
    ),
    agent=planner
)

write_task = Task(
    description=(
        "Using the content plan, write an educational article on {topic}. "
        "Explain technical terms, use meaningful headings, and distinguish "
        "well-established facts from opinions or predictions."
    ),
    expected_output="A well-structured Markdown article of approximately 700–900 words.",
    agent=writer,
    markdown=True
)

def editorial_guardrail(output):
    text = output.raw.strip()
    if len(text.split()) < 250:
        return (False, "The article is too short. Expand the explanation to at least 250 words.")
    if not any(mark in text for mark in ["##", "#"]):
        return (False, "Use Markdown headings to structure the article.")
    return (True, text)

edit_task = Task(
    description=(
        "Edit the article for clarity, logical flow, grammar and consistency. "
        "Do not add facts or references that are not supported by the draft. "
        "Flag statements that need independent verification."
    ),
    expected_output="A polished Markdown article with a short editorial note listing any claims to verify.",
    agent=editor,
    markdown=True,
    guardrail=editorial_guardrail,
    guardrail_max_retries=2
)

## Step 3 — Assemble and run the Crew

`Process.sequential` makes the task order explicit. `verbose=True` helps inspect execution during the lab.

In [ ]:
article_crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan_task, write_task, edit_task],
    process=Process.sequential,
    verbose=True
)

topic = "Agentic AI in higher education"
result = article_crew.kickoff(inputs={"topic": topic})

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result.raw))

## Step 4 — Observe the workflow

- Which agent ran first?
- What information passed from Planner to Writer?
- What did the Editor change?
- Did the guardrail accept the final output?

## Try it yourself
1. Change the topic to a domain of your choice.
2. Add a Fact Checker agent and task between Writer and Editor.
3. Compare a short article with a longer article.
4. Run once with one agent doing all three jobs and compare quality, time and cost.

**Important:** Without a search tool, the Planner cannot verify current news or latest developments. Treat the output as a draft, not a sourced research report.